# FashionMNIST Image Classifier

An MLP image classifier for FashionMNIST, built step by step. Run the cells in
order from top to bottom.


## 0. Setup

Install the third-party dependencies into the current kernel. `torch` and
`torchvision` are usually already available, but `torchmetrics` often is not, so
install it here to make the notebook self-contained. If you have already
installed these, this cell is a quick no-op.


In [ ]:
# Install dependencies into the kernel that is running this notebook.
# %pip installs into the *current* kernel, which avoids the common
# "installed in the wrong environment" problem with plain !pip.
%pip install -q torch torchvision torchmetrics

## 1. Load the data

Load FashionMNIST through TorchVision. We build a training/validation dataset
and a separate test dataset, then split the training/validation data into
55,000 training and 5,000 validation samples using a fixed seed of 42.


In [1]:
import torch
from torch.utils.data import random_split, DataLoader
import torchvision
from torchvision import transforms

# Turn the PIL images into float tensors in the range [0, 1].
transform = transforms.ToTensor()

# Full training set (60,000 images) that we split into train + validation.
train_val_dataset = torchvision.datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

# Separate test set (10,000 images).
test_dataset = torchvision.datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

# Split 60,000 -> 55,000 train / 5,000 validation, seeded for reproducibility.
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    train_val_dataset,
    [55_000, 5_000],
    generator=generator,
)

print(f"Train samples:      {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")


100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 26.4M/26.4M [00:03<00:00, 6.72MB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 29.5k/29.5k [00:00<00:00, 198kB/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 4.42M/4.42M [00:02<00:00, 1.90MB/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 5.15k/5.15k [00:00<00:00, 11.1MB/s]

Train samples:      55000
Validation samples: 5000
Test samples:       10000


## 2. Create the DataLoaders

Wrap each dataset in a `DataLoader` with a batch size of 32. The training loader
is shuffled (with a seeded generator for a reproducible batch order); the
validation and test loaders are not shuffled.


In [2]:
BATCH_SIZE = 32

# Seed the generator that drives the training loader's shuffling.
loader_generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Batch size:         {BATCH_SIZE}")
print(f"Train batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")


Batch size:         32
Train batches:      1719
Validation batches: 157
Test batches:       313


## 3. Sample the data

Inspect the first training sample to confirm the image shape (`(1, 28, 28)`)
and dtype (`torch.float32`), and list the 10 FashionMNIST class names.


In [3]:
# First (image, label) pair from the training data.
x_sample, y_sample = train_dataset[0]

print(f"x sample shape: {tuple(x_sample.shape)}")   # (1, 28, 28)
print(f"x sample dtype: {x_sample.dtype}")           # torch.float32

# train_dataset and val_dataset are Subsets of the same underlying dataset,
# so the class names live on that underlying FashionMNIST object.
class_names = train_dataset.dataset.classes

print(f"\nNumber of classes: {len(class_names)}")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

print(f"\nFirst sample label: {y_sample} -> {class_names[y_sample]}")


x sample shape: (1, 28, 28)
x sample dtype: torch.float32

Number of classes: 10
  0: T-shirt/top
  1: Trouser
  2: Pullover
  3: Dress
  4: Coat
  5: Sandal
  6: Shirt
  7: Sneaker
  8: Bag
  9: Ankle boot

First sample label: 9 -> Ankle boot


## 4. Build the model

`ImageClassifier` is a flexible MLP. Its constructor takes the flattened
`input_size` (28 × 28 = 784), a list of `hidden_sizes`, and `num_classes` (10),
and assembles an `nn.Sequential`. We use hidden layers `[300, 100]` and
`CrossEntropyLoss` as the loss function.


In [4]:
import torch.nn as nn


class ImageClassifier(nn.Module):
    """A configurable fully-connected (MLP) image classifier.

    Args:
        input_size:   Number of input features after flattening (784 here).
        hidden_sizes: Widths of the hidden layers; each entry adds a
                      Linear + ReLU block.
        num_classes:  Number of output classes (10 for FashionMNIST).
    """

    def __init__(self, input_size, hidden_sizes, num_classes):
        super().__init__()

        # Flatten (batch, 1, 28, 28) -> (batch, 784) then stack hidden blocks.
        layers = [nn.Flatten()]
        in_features = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            in_features = hidden_size
        # Output layer -> class logits (no activation; CrossEntropyLoss handles
        # the softmax internally).
        layers.append(nn.Linear(in_features, num_classes))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


input_size = 28 * 28
hidden_sizes = [300, 100]
num_classes = 10

model = ImageClassifier(input_size, hidden_sizes, num_classes)
loss_fn = nn.CrossEntropyLoss()

print(model)
print(f"\nLoss function: {loss_fn}")


ImageClassifier(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=300, bias=True)
    (2): ReLU()
    (3): Linear(in_features=300, out_features=100, bias=True)
    (4): ReLU()
    (5): Linear(in_features=100, out_features=10, bias=True)
  )
)

Loss function: CrossEntropyLoss()


## 5. Optimizer and training

Use SGD (`lr=0.01`) to update the parameters. The `train()` function ties
together the model, loaders, loss function, and optimizer, runs for 20 epochs,
and reports train/validation loss and accuracy each epoch. Accuracy is measured
with torchmetrics `MulticlassAccuracy` (`average="micro"`). `train()` returns a
`history` of the per-epoch metrics.


In [5]:
from torchmetrics.classification import MulticlassAccuracy

# SGD optimizer with a learning rate of 0.01.
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Micro-averaged accuracy = overall correct / total.
train_accuracy = MulticlassAccuracy(num_classes=num_classes, average="micro")
val_accuracy = MulticlassAccuracy(num_classes=num_classes, average="micro")


def train(model, train_loader, val_loader, loss_fn, optimizer,
          train_accuracy, val_accuracy, num_epochs=20):
    """Train the model and report metrics each epoch.

    Returns a history dict with per-epoch train/validation loss and accuracy.
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, num_epochs + 1):
        # --- Training pass ---
        model.train()
        train_accuracy.reset()
        running_loss, seen = 0.0, 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            seen += labels.size(0)
            train_accuracy.update(outputs, labels)

        train_loss = running_loss / seen
        train_acc = train_accuracy.compute().item()

        # --- Validation pass ---
        model.eval()
        val_accuracy.reset()
        running_loss, seen = 0.0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                outputs = model(images)
                loss = loss_fn(outputs, labels)

                running_loss += loss.item() * images.size(0)
                seen += labels.size(0)
                val_accuracy.update(outputs, labels)

        val_loss = running_loss / seen
        val_acc = val_accuracy.compute().item()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch:2d}/{num_epochs}  "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    return history


history = train(
    model, train_loader, val_loader, loss_fn, optimizer,
    train_accuracy, val_accuracy, num_epochs=20,
)


ModuleNotFoundError: No module named 'torchmetrics'

## 6. Evaluate the model

Run the model over the validation loader, take the `argmax` of the logits as the
predicted class, and compare against the true labels to see which predictions
were correct.


In [ ]:
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images)              # (batch, num_classes)
        preds = logits.argmax(dim=1)         # predicted class per image

        all_preds.append(preds)
        all_labels.append(labels)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

# Which predictions were correct?
correct = all_preds == all_labels
num_correct = correct.sum().item()
total = correct.numel()

print(f"Correct predictions: {num_correct} / {total}")
print(f"Validation accuracy: {num_correct / total:.4f}")

print(f"\nFirst 10 predictions: {all_preds[:10].tolist()}")
print(f"First 10 labels:      {all_labels[:10].tolist()}")
print(f"First 10 correct?:    {correct[:10].tolist()}")


## 7. Softmax probabilities and model size

Apply `F.softmax` to a sample's logits, pull the top 4 probabilities and their
class indices with `torch.topk`, and count the model's total parameters with
`numel()` (266,610).


In [ ]:
import torch.nn.functional as F

# Take one batch from the validation loader and use its first sample.
images, labels = next(iter(val_loader))
model.eval()
with torch.no_grad():
    logits = model(images)

y_logits = logits[0]                 # logits for the first sample

# Softmax turns logits into class probabilities.
probs = F.softmax(y_logits, dim=0)

# Top 4 probabilities and the classes they belong to.
top4_values, top4_indices = torch.topk(probs, 4)

print(f"Softmax probabilities: {probs.tolist()}")
print(f"\nTop 4 values:  {top4_values.tolist()}")
print(f"Top 4 indices: {top4_indices.tolist()}")
print(f"Top 4 classes: {[class_names[i] for i in top4_indices.tolist()]}")

# Total number of parameters in the model.
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal model parameters: {total_params:,}")   # 266,610
